In [1]:
import trino
import pandas as pd
from datetime import datetime

In [2]:
conn = trino.dbapi.connect(
    host="trino",
    port=8080,
    user="jupyter",
    catalog="iceberg"
)

In [3]:
cur = conn.cursor()

In [4]:
ctas_query = f"""
CREATE TABLE IF NOT EXISTS serving_db.sma7
WITH (
    format = 'PARQUET',
    location = 's3a://crypto-data-lake/serving_zone/sma7'
) AS
select 
    *,
    round((avg(close_price) over(order by group_id rows between 6 preceding and current row)), 2) as sma7
from serving_db.klines
"""
cur.execute(ctas_query)

In [5]:
cur.execute("SELECT count(*) FROM serving_db.sma7").fetchall()

[[96]]

In [7]:
cur.execute("SELECT * FROM iceberg.serving_db.klines")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head(10)

,group_id,group_date,open_time,open_price,high_price,low_price,close_price,volume,close_time
0,1948896,2025-08-01 00:00:00,1754006400328945,115764.07,115829.46,115308.55,115313.01,302.16,1754007299467573
1,1948897,2025-08-01 00:15:00,1754007300010950,115313.01,115933.00,115313.00,115800.01,450.54,1754008199447993
2,1948898,2025-08-01 00:30:00,1754008200077603,115800.00,115800.00,115423.87,115517.98,184.43,1754009099900832
3,1948899,2025-08-01 00:45:00,1754009100223687,115517.99,115527.53,114313.13,115427.27,1589.76,1754009999974074
4,1948900,2025-08-01 01:00:00,1754010000363342,115427.27,115609.99,114600.00,114649.90,681.88,1754010899995166
5,1948901,2025-08-01 01:15:00,1754010900041356,114649.90,115271.13,114638.65,115190.38,448.24,1754011799896691
6,1948902,2025-08-01 01:30:00,1754011800063081,115190.37,115413.91,115000.00,115296.45,267.39,1754012699912234
7,1948903,2025-08-01 01:45:00,1754012700223622,115296.46,115407.71,115060.11,115331.86,258.53,1754013599937883
8,1948904,2025-08-01 02:00:00,1754013600005133,115328.67,115600.00,115221.07,115600.00,164.01,1754014499986628
9,1948905,2025-08-01 02:15:00,1754014500063435,115600.00,115810.71,115511.60,115619.94,168.41,1754015399984518


In [8]:
cur.execute("SHOW COLUMNS FROM iceberg.serving_db.klines")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head(10)

,Column,Type,Extra,Comment
0,group_id,bigint,,
1,group_date,varchar,,
2,open_time,bigint,,
3,open_price,double,,
4,high_price,double,,
5,low_price,double,,
6,close_price,double,,
7,volume,double,,
8,close_time,bigint,,


In [9]:
cur.execute("""
SELECT
  from_unixtime(open_time / 1000000) AS "time",
  open_price,
  high_price,
  low_price,
  close_price,
  volume
FROM iceberg.serving_db.klines
ORDER BY "time"
""")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head(10)

,time,open_price,high_price,low_price,close_price,volume
0,2025-08-01 00:00:00+00:00,115764.07,115829.46,115308.55,115313.01,302.16
1,2025-08-01 00:15:00+00:00,115313.01,115933.00,115313.00,115800.01,450.54
2,2025-08-01 00:30:00+00:00,115800.00,115800.00,115423.87,115517.98,184.43
3,2025-08-01 00:45:00+00:00,115517.99,115527.53,114313.13,115427.27,1589.76
4,2025-08-01 01:00:00+00:00,115427.27,115609.99,114600.00,114649.90,681.88
5,2025-08-01 01:15:00+00:00,114649.90,115271.13,114638.65,115190.38,448.24
6,2025-08-01 01:30:00+00:00,115190.37,115413.91,115000.00,115296.45,267.39
7,2025-08-01 01:45:00+00:00,115296.46,115407.71,115060.11,115331.86,258.53
8,2025-08-01 02:00:00+00:00,115328.67,115600.00,115221.07,115600.00,164.01
9,2025-08-01 02:15:00+00:00,115600.00,115810.71,115511.60,115619.94,168.41


In [11]:
cur.execute("""
SELECT
  from_unixtime(open_time / 1000000) AS "time",
  sma7
FROM iceberg.serving_db.sma7
ORDER BY "time"
""")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head(10)

,time,sma7
0,2025-08-01 00:00:00+00:00,115313.01
1,2025-08-01 00:15:00+00:00,115556.51
2,2025-08-01 00:30:00+00:00,115543.67
3,2025-08-01 00:45:00+00:00,115514.57
4,2025-08-01 01:00:00+00:00,115341.63
5,2025-08-01 01:15:00+00:00,115316.43
6,2025-08-01 01:30:00+00:00,115313.57
7,2025-08-01 01:45:00+00:00,115316.26
8,2025-08-01 02:00:00+00:00,115287.69
9,2025-08-01 02:15:00+00:00,115302.26
